<a href="https://colab.research.google.com/github/WilsonRomao/estoque_farmacia_sesau/blob/main/backend/pipeline/analise_relatorio_geral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports e constantes

In [26]:
import pandas as pd
# importar dados do drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
base_path = '/content/drive/MyDrive/UFMS - SISTEMAS DE INFORMAÇÃO/PET SAUDE/'



df = pd.read_excel(base_path+"/bronze/Toda a rede.xls", header=None)

# print(df)



In [1]:
import os
import pandas as pd

# Caminho para a pasta contendo os arquivos Excel
BASE_PATH = os.getcwd()
# subir um nível para acessar a pasta "uploads"
UPLOAD_PATH = os.path.dirname(BASE_PATH)
EXCEL_FOLDER = os.path.join(UPLOAD_PATH, 'uploads')


## Funções

### Extração de dados

In [54]:
# Extrair os dados de cada arquivo Excel e armazenar em um DataFrame
def extract_data_from_excels():
    data_frames = []
    for file_name in os.listdir(EXCEL_FOLDER):
        if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
            file_path = os.path.join(EXCEL_FOLDER, file_name)
            df = pd.read_excel(file_path, header=None)  # Ler o arquivo Excel sem considerar a primeira linha como cabeçalho
            data_frames.append(df)  
    # Concatenar todos os DataFrames em um único DataFrame
    final_df = pd.concat(data_frames, ignore_index=True)
    return final_df

### Transformação

In [ ]:
# função para encontrar os estabelecimentos de saúde:
def select_estabelecimentos_saude(df):
    # Filtrar as linhas que contêm "Estabelecimento de Saúde:" na coluna 1
    estabelecimentos =df[df[1].astype(str).str.contains('Estabelecimento', case=True, na=False, regex=True)].dropna(axis=1, how='all').copy()

    #cria uma nova coluna "row" para armazenar o número da linha de cada estabelecimento
    estabelecimentos["row"] = estabelecimentos.index

    # Extrair o nome do estabelecimento de saúde da coluna 1
    estabelecimentos[1] = estabelecimentos[1].str.replace("Estabelecimento de Saúde:", "", case=False).str.strip()

    # Eliminar duplicatas
    estabelecimentos = estabelecimentos.drop_duplicates(subset=[1])    

    # substituir caracter "/" por "-" para evitar problemas para identificar o direito do arquivo posteriormente
    estabelecimentos[1] = estabelecimentos[1].str.replace("/", "-")

    # resetar o índice do DataFrame para que ele comece do zero
    estabelecimentos = estabelecimentos.reset_index(drop=True)

    return estabelecimentos

RangeIndex(start=0, stop=47127, step=1)

## Exploração

### Informações iniciais

In [6]:
df = extract_data_from_excels()
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,05/12/2025,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,Posição de,NaN,- Município,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,Estoque em,NaN,05/12/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Produto,NaN,Unidade,Fabricante,Localização,NaN,Programa de Saúde,NaN,Validade,Lote,Fator,NaN,NaN,Bloq,NaN,NaN,Qtde,NaN,Valor
4,NaN,Estabelecimento de Saúde: SESAU DAF ALMOX BÁS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47132 entries, 0 to 47131
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       0 non-null      float64
 1   1       34103 non-null  object 
 2   2       154 non-null    object 
 3   3       13025 non-null  object 
 4   4       13025 non-null  object 
 5   5       13025 non-null  object 
 6   6       2 non-null      object 
 7   7       13025 non-null  object 
 8   8       2 non-null      object 
 9   9       13025 non-null  object 
 10  10      13025 non-null  object 
 11  11      13025 non-null  object 
 12  12      16838 non-null  float64
 13  13      13024 non-null  object 
 14  14      1 non-null      object 
 15  15      1 non-null      object 
 16  16      13024 non-null  object 
 17  17      1 non-null      object 
 18  18      29861 non-null  object 
 19  19      1 non-null      object 
dtypes: float64(2), object(18)
memory usage: 7.2+ MB


In [196]:
df

,1,2,3,4,5,7,9,10,11,12,13,16,18
0,Produto,NaN,Unidade,Fabricante,Localização,Programa de Saúde,Validade,Lote,Fator,NaN,NaN,NaN,NaN
1,Estabelecimento de Saúde: SESAU DAF ALMOX BÁS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fonte de,MUNICIPAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BR0267618U0042 CARBAMAZEPINA 200 MG COMPRIMID...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47122,"BR0442230 VORTIOXETINA, BROMIDATO 5 MG COMPRIMIDO",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47123,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,31/05/2026,4J6214,1,NaN,N,20,"0,00"
47124,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,0
47125,Total Relatório:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9944075.0,NaN,NaN,NaN


### Tratando valores nulos

In [8]:
# Dropar colunas irrelevantes pelo indice (0, 6, 8,14,15,17,19))
df.drop(df.columns[[0, 6, 8,14,15,17,19]], axis=1, inplace=True)
df.head()


,1,2,3,4,5,7,9,10,11,12,13,16,18
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Produto,NaN,Unidade,Fabricante,Localização,Programa de Saúde,Validade,Lote,Fator,NaN,NaN,NaN,NaN
4,Estabelecimento de Saúde: SESAU DAF ALMOX BÁS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# Dropar linhas completamente vazias
df.dropna(how='all', inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,1,2,3,4,5,7,9,10,11,12,13,16,18
0,Produto,NaN,Unidade,Fabricante,Localização,Programa de Saúde,Validade,Lote,Fator,NaN,NaN,NaN,NaN
1,Estabelecimento de Saúde: SESAU DAF ALMOX BÁS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fonte de,MUNICIPAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BR0267618U0042 CARBAMAZEPINA 200 MG COMPRIMID...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0


### Compreendendo as colunas

A coluna 12 contém a quantidade total por medicamento e na ultima a quantidade total do relatório.

In [34]:
df[df[12].notna()]

,1,2,3,4,5,7,9,10,11,12,13,16,18
4,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
6,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
10,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
15,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,600.0,NaN,NaN,0
19,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24630.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47115,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,0
47118,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,0
47121,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,0
47124,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,0


As colunas 13 e 16 contem dados na mesma linha

In [37]:
df[df[13].notna()]

,1,2,3,4,5,7,9,10,11,12,13,16,18
14,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,31/08/2027,50031821,1,NaN,N,600,"0,00"
17,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,DST/AIDS,31/01/2027,DGYG,1,NaN,N,3.900,"0,00"
18,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,DST/AIDS,31/03/2027,DSMP,1,NaN,N,20.730,"0,00"
21,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,31/03/2027,25030323,1,NaN,N,3.540,"8.460,60"
22,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,30/04/2027,25040460,1,NaN,N,12.330,"21.294,90"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47111,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/04/2026,4D5864,1,NaN,N,3,"0,00"
47114,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/06/2026,4F3564,1,NaN,N,14,"0,00"
47117,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,31/03/2026,4B3709,1,NaN,N,30,"0,00"
47120,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/06/2026,4E5659,1,NaN,N,40,"0,00"


In [38]:
df[df[16].notna()]

,1,2,3,4,5,7,9,10,11,12,13,16,18
14,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,31/08/2027,50031821,1,NaN,N,600,"0,00"
17,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,DST/AIDS,31/01/2027,DGYG,1,NaN,N,3.900,"0,00"
18,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,DST/AIDS,31/03/2027,DSMP,1,NaN,N,20.730,"0,00"
21,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,31/03/2027,25030323,1,NaN,N,3.540,"8.460,60"
22,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,30/04/2027,25040460,1,NaN,N,12.330,"21.294,90"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47111,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/04/2026,4D5864,1,NaN,N,3,"0,00"
47114,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/06/2026,4F3564,1,NaN,N,14,"0,00"
47117,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,31/03/2026,4B3709,1,NaN,N,30,"0,00"
47120,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/06/2026,4E5659,1,NaN,N,40,"0,00"


In [39]:
df[df[18].notna()]

,1,2,3,4,5,7,9,10,11,12,13,16,18
4,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
6,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
10,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
14,NaN,NaN,COMP.,FIOCRUZ,ARMARIO DE ANTIRETROVIRAIS,ASSISTÊNCIA FARMACÊUTICA,31/08/2027,50031821,1,NaN,N,600,"0,00"
15,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,600.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47118,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,0
47120,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,30/06/2026,4E5659,1,NaN,N,40,"0,00"
47121,Total:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,0
47123,NaN,NaN,COMP.,EMS SGMA PHARMA,PRATELEIRAS,ASSISTÊNCIA FARMACÊUTICA BÁSICA,31/05/2026,4J6214,1,NaN,N,20,"0,00"


### Listar os Estabelecimentos de Saúde



In [157]:
type(df)

pandas.core.frame.DataFrame

In [ ]:
estabelecimentos = select_estabelecimentos_saude(df)
estabelecimentos

,1,row
0,SESAU DAF ALMOX BÁSICO CAMPO GRANDE,1
1,CTA - CENTRO DE TESTAGEM E ACONSELHAMENTO CAMP...,7
2,ESTABELECIMENTO PENAL DE SEGURANÇA MÁXIMA- JAI...,376
3,MODULO DE SAUDE DO COMPLEXO PENITENCIARIO-A DE...,413
4,PENITENCIARIA ESTADUAL MASC REGIME FECHADO GAM...,850
...,...,...
84,SESAU USF SAO FRANCISCO,43926
85,SESAU USF SEBASTIAO LUIZ NOGUEIRA LOS ANGELES,44696
86,SESAU USF VILA CARVALHO,45287
87,SESAU USF VILA CORUMBA,46014


In [ ]:
# Função que recebe o dataframe de medicamento, o df e o i e retorna a quantidade do medicamento
def select_quantidade(df, i):
    inicio = medicamentos.loc[i,'row_medicamento']
    fim = medicamentos.loc[i+1,'row_medicamento'] if i+1 < len(medicamentos) else len(df)
    temp_df = df.iloc[inicio:fim]

    quantidade = temp_df[temp_df[1].astype(str).str.contains("Total", case=False, na=False, regex=True)].dropna(axis=1, how='all').copy()
    return quantidade[12]

In [ ]:
def select_medicamentos(estabelecimentos, df,i):

    inicio = estabelecimentos.loc[i,'row']
    fim = estabelecimentos.loc[i+1,'row'] if i+1 < len(estabelecimentos) else len(df)
    temp_df = df.iloc[inicio:fim]
    medicamentos = temp_df[temp_df[1].astype(str).str.contains("BR", case=False, na=False, regex=True)].dropna(axis=1, how='all').copy()

    medicamentos["row_medicamento"] = medicamentos.index
    
    medicamentos = medicamentos.reset_index(drop=True)

    return medicamentos
        
    

In [257]:
med = select_medicamentos(estabelecimentos, df, 1)
for i in med.index:
    med["quantidade"] = []
    med["quantidade"] = select_quantidade(med, df, i)
med

<class 'pandas.core.frame.DataFrame'>


ValueError: Length of values (0) does not match length of index (124)

49    0.0
Name: 12, dtype: float64

A ideia agora é utilizar o indice das unidades de saúde para limitar a busca por medicamentos:

o índice da unidade(atual) é a posição inicial da busca;
o índice da udidade(proxima)-1 é a posição final da busca;

In [ ]:
for estabelecimento in estabelecimentos:
    print(estabelecimento)

AttributeError: 'str' object has no attribute 'key'

# Extraindo informações de cada unidade

Extraindo dados dos medicamentos

In [113]:
df.iloc[estabelecimentos.keys[0]:estabelecimentos.keys[1]]

TypeError: 'builtin_function_or_method' object is not subscriptable

In [ ]:
medicamentos_df = dataframe[dataframe['Produto'].astype(str).str.contains('BR', case=True, na=False, regex=True)].dropna(axis=1, how='all').reset_index(drop=True)
medicamentos_df = medicamentos_df.rename(columns={'Produto': 'nome_medicamento'})
medicamentos_df[['codigo_medicamento','nome_medicamento']] = medicamentos_df['nome_medicamento'].str.split(' ', n=1, expand=True)


display(medicamentos_df)

In [ ]:
quantidade_df = dataframe[dataframe['Produto'].astype(str).str.contains('Total', case=True, na=False, regex=True)].dropna(axis=1, how='all').reset_index(drop=True)

quantidade_df = quantidade_df.drop(columns=['Produto',18])
quantidade_df = quantidade_df.rename(columns={12: 'quantidade'})

display(quantidade_df)

In [ ]:
estoque = pd.merge(medicamentos_df, quantidade_df, left_index=True, right_index=True, how='inner')

estoque = estoque[['codigo_medicamento','nome_medicamento','quantidade']]
estoque

In [ ]:
# save estoue in a json named estoque_processado_alves_pereira
estoque.to_json(base_path + '/estoque_processado_alves_pereira.json', orient='records')
# # SAVA A CSV WITH THE SAME NAME
# estoque.to_csv(base_path + '/estoque_processado_alves_pereira.csv', index=False)